# 05. 원본과 연결되지 않게 필요한 부분 고르기

먼저 결과를 예상하고, 코드를 직접 실행한 뒤 실제 값과 결과 모양을 확인함. 이 파일은 위에서 아래로 차례로 실행하면 됨.


### 먼저 알아둘 배열 말 네 가지

`np.array([[1, 2], [3, 4]])`는 NumPy가 다루는 2차원 숫자 배열임. NumPy의 정확한 이름은 **ndarray**이며, 같은 저장 형식의 값을 여러 축에 놓는 자료구조임.

- **shape(결과 모양)**: 각 축의 칸 수임. 위 배열은 2행 2열이라 `(2, 2)`이며 `a.shape`로 확인함.
- **dimension(차원)**: 축의 개수임. 위 배열은 축이 두 개라 `a.ndim == 2`임. 차원과 전체 원소 수는 다름.
- **axis(축)**: 값의 위치가 변하는 방향임. 2차원 표에서 `axis=0`은 행을 따라 내려가는 방향, `axis=1`은 열을 따라 옆으로 가는 방향임. 축 번호만으로 실제 센서 의미나 단위까지 정해지지는 않음.
- **dtype(저장 형식)**: 각 값을 정수·실수·문자 중 어떤 방식으로 저장하는지 나타냄. `a.dtype`으로 확인하며, 값의 단위나 현실 의미를 보장하지 않음.

계산에서 **숫자 하나로 다루는 값**을 scalar(스칼라)라고 부름. 다만 Python 숫자 `10`은 `.shape`가 없고, `np.array(10)`은 shape `()`인 0차원 ndarray이며, `np.float64(10)` 같은 NumPy scalar도 별도 타입이라 세 표현이 완전히 같지는 않음. `np.asarray(value).shape`로 배열 관점의 모양을 확인함. `True`와 `False`는 **Boolean(참·거짓)** 값임. Boolean 배열을 **mask(마스크)**로 쓰면 `True`인 위치만 고를 수 있지만, mask의 모양과 어느 축을 고르는지는 먼저 확인해야 함.

먼저 결과를 예측하고, 코드를 직접 실행한 뒤 실제 값과 모양(shape)을 확인함.


In [ ]:
import json
import numpy as np

오류_예시를_실행할지 = False
연습_데이터 = json.loads(r'''{"slice_selection":{"data":[[68,0.31,101.2],[71,0.36,100.8],[74,0.44,102.1],[77,0.53,103.4],[79,0.58,104]]}}''')

def 배열_확인(name, value):
    array = np.asarray(value)
    print(f"{name}: value={array}, shape={array.shape}, dtype={array.dtype}")
    return array

print(f"NumPy version: {np.__version__}")
if 연습_데이터:
    print(f"이 단원에서 쓸 연습 데이터: {list(연습_데이터)}")


## 원본과 연결되지 않게 필요한 부분 고르기

**핵심 질문:** 오류 없이 실행돼도 왜 엉뚱한 센서나 시간 구간을 고를 수 있을까?


#### 먼저 생각

**위치와 원본 연결 용어부터 확인함.** **index(인덱스)**는 0부터 세는 위치 번호임. `a=np.array([10,20,30])`에서 `a[1]`은 두 번째 위치의 값 `20`을 고름. 2차원 배열 `table[row_index, column_index]`에서는 앞 번호가 행, 뒤 번호가 열을 고르며 두 번호 모두 0부터 셈. index는 실제 값 `20`과 다르고, `a[1:3]`처럼 여러 위치를 연속으로 고르는 **slice(슬라이스)**와도 다름. **view**는 원본과 같은 메모리를 볼 수 있는 선택 결과, **copy**는 독립된 새 배열임. **mutation**은 기존 배열 값을 직접 바꾸는 일임. 예를 들어 `a=np.array([1,2,3]); b=a[1:]` 뒤 `b[0]=9`가 `a`도 바꾸면 view임. 값이 같아 보이는 것과 메모리를 공유하는 것은 다르며 `np.shares_memory(a, b)`로 확인함.
**핵심 질문**: 오류 없이 실행돼도 왜 엉뚱한 센서나 시간 구간을 고를 수 있을까?

**실행 전 예측**

시간×센서 배열에서 `data[2]`, `data[:,1]`, `data[1:4,0:2]`의 의미와 shape를 예측하셈.

> 내 예측(값·shape·조건·단위): `TODO`


#### 개념

**왜 배우는지와 주의할 점**

`a[row, column]`, start 포함·stop 제외. basic slicing은 view일 수 있고, 원본 보호가 필요하면 `.copy()`를 명시함.

코드를 볼 때는 무엇을 계산하는지, 결과 모양이 어떤지, 원래 배열이 바뀌는지를 함께 확인함.


### 개념 도식




In [ ]:
sensor = np.array(연습_데이터['slice_selection']['data'], dtype=float)
original = sensor.copy()
view = sensor[1:3, :2]
view[0,0] = -999
print(f"view 수정이 원본에 반영: {sensor[1,0] == -999}")
sensor = original
safe = sensor[1:3,:2].copy()
safe[0,0] = -999
print(f"copy 수정 뒤 원본 보존: {sensor[1,0] != -999}")


### 실행 뒤 해석

방금 출력에서 실제 값, 결과 모양(shape), 저장 형식(dtype), 원래 배열이 바뀌었는지를 한 문장으로 정리하셈. 예상과 다르면 어느 입력이나 축을 다시 확인할지도 적으셈.


### 🧪 학생 문제

0부터 세는 index 기준으로 시간 index 1, 2, 3 행과 센서 index 0, 1 열을 고르되, 이후 선택본 수정이 원본에 영향을 주면 안 됨. 선택식·shape·copy 필요성을 제안하셈.

1. 결과·shape·조건을 먼저 쓰셈.
2. 아래 TODO 셀에 자기 코드를 작성하셈.
3. 출력은 `실제 값 / 결과 모양(shape) / 저장 형식(dtype) / 오류`로 기록하셈.


In [ ]:
# 직접 작성할 부분
problem_data = np.arange(24).reshape(6, 4)
# TODO: 1~3번 시간 행과 0~1번 센서 열을 원본과 독립적으로 선택하셈.
selection = None
predicted_shape = None
print(f"원본 shape: {problem_data.shape} 예상 선택 shape: {predicted_shape}")


### 풀이 전 자기 점검

- 결과 shape가 예측과 같은가?
- 계산 목적과 연산이 일치하는가?
- 단위·원본 변경·경계 조건을 확인했는가?


### 다른 조건에서 교정·재시도

완성 답안을 복사하지 말고, 조건이 바뀐 입력에서 판단 절차를 다시 적용함. 아래 셀은 다른 입력의 안전한 실행 결과임.


In [ ]:
# 조건을 바꿔 다시 확인함
retry_data = np.arange(15).reshape(3, 5)  # 이번에는 행=센서, 열=시간
retry_copy = retry_data[:, 1:4].copy()
retry_copy[0, 0] = -1
assert retry_data[0, 1] != -1
print(f"축 의미 변경 뒤 원본 불변: {retry_data.shape} {retry_copy.shape}")


#### 응용

축 의미→범위→예상 shape→원본 변경 위험 순서로 코드 전에 설계함.

> 새 조건에서 유지할 판단과 보류할 해석: `TODO`


### 자기 점검

- [ ] 실행 전 결과·shape·조건을 예측함.
- [ ] 실제 출력·shape·dtype 또는 오류를 기록함.
- [ ] 오류를 원인과 유형으로 설명함.
- [ ] 최소 수정 후 다른 조건에서 재실행함.
- [ ] 계산 가능성과 해석 가능성을 구분함.

내가 아직 증명하지 못한 것: `TODO`
